# Oral screen shift: GPU run

Fine-tunes the ResNet models and saves their predictions. Before running: **Runtime > Change runtime type > T4 GPU**.

Run the cells top to bottom. If Colab disconnects, run them all again: finished models are skipped.

## 1. Get the code and connect Google Drive

Drive only stores the trained models and predictions (about 1.5 GB), so a disconnect does not lose them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf oral-screen-shift
!git clone -q https://github.com/sachchi-sapiens/oral-screen-shift.git
%cd /content/oral-screen-shift
!pip install -q timm

!mkdir -p /content/drive/MyDrive/oral-screen-shift/checkpoints /content/drive/MyDrive/oral-screen-shift/preds results
!ln -sfn /content/drive/MyDrive/oral-screen-shift/checkpoints checkpoints
!cp -n results/preds/*.npy /content/drive/MyDrive/oral-screen-shift/preds/ 2>/dev/null; rm -rf results/preds
!ln -sfn /content/drive/MyDrive/oral-screen-shift/preds results/preds

import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

## 2. Check the code works here

In [ ]:
!python -m pytest -q

## 3. Download the four datasets (about 10 GB, onto Colab's own disk)

In [ ]:
for site in ['smartom', 'china', 'karnataka', 'code']:
    !python scripts/download.py --site {site} --out /content/data

## 4. Make small copies of the photos so training is fast

In [ ]:
!python scripts/shrink.py --data /content/data --out /content/data_small

## 5. Train the baseline ResNet-50 (6 models)

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage resnet

## 6. Train the ResNet-50 with colour augmentation (9 models)

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage resnet_color

## 7. Pack the predictions

This writes `preds.zip` to the `oral-screen-shift` folder in your Google Drive. Download it and unzip it into `results/preds/` in the project folder on your laptop.

In [ ]:
!cd /content/drive/MyDrive/oral-screen-shift && zip -q -r preds.zip preds && ls -lh preds.zip && ls preds | wc -l